In [1]:
import os
from tfx.orchestration.beam.beam_dag_runner import BeamDagRunner
from tfx.orchestration import metadata, pipeline

# nama pipeline dan pengaturan path
PIPELINE_NAME = "mualim-fth-pipeline"
DATA_ROOT = 'data'
PIPELINE_ROOT = os.path.join('pipelines', PIPELINE_NAME)
METADATA_PATH = os.path.join('metadata', PIPELINE_NAME, 'metadata.db')
SERVING_MODEL_DIR = os.path.join('serving_model', PIPELINE_NAME)

In [2]:
%%writefile modules/transform.py
import tensorflow as tf
import tensorflow_transform as tft

CATEGORICAL_FEATURES = {
    'gender': 2, 'SeniorCitizen': 2, 'Partner': 2,
    'StreamingTV': 3, 'PhoneService': 2, 'InternetService': 3,
    'PaperlessBilling': 2
}
NUMERIC_FEATURES = ['tenure', 'MonthlyCharges', 'TotalCharges']
LABEL_KEY = 'Churn'

def transformed_name(key):
    return key + '_xf'

def preprocessing_fn(inputs):
    outputs = {}
    
    for key in CATEGORICAL_FEATURES:
        dim = CATEGORICAL_FEATURES[key]
        int_value = tft.compute_and_apply_vocabulary(inputs[key], top_k=dim + 1)
        outputs[transformed_name(key)] = tf.cast(int_value, tf.int64)
        
    for key in NUMERIC_FEATURES:
        if inputs[key].dtype == tf.string:
            tensor_float = tf.strings.to_number(inputs[key], out_type=tf.float32)
            outputs[transformed_name(key)] = tft.scale_to_z_score(tensor_float)
        else:
            outputs[transformed_name(key)] = tft.scale_to_z_score(inputs[key])
            
    outputs[transformed_name(LABEL_KEY)] = tf.cast(
        tft.compute_and_apply_vocabulary(inputs[LABEL_KEY], top_k=2),
        tf.int64
    )
    
    return outputs

Overwriting modules/transform.py


In [3]:
%%writefile modules/components.py
import os
import tensorflow_model_analysis as tfma
from tfx.components import (
    CsvExampleGen, StatisticsGen, SchemaGen, ExampleValidator, 
    Transform, Trainer, Tuner, Evaluator, Pusher
)
from tfx.proto import example_gen_pb2, trainer_pb2, pusher_pb2
from tfx.types import Channel
from tfx.dsl.components.common.resolver import Resolver
from tfx.types.standard_artifacts import Model, ModelBlessing
from tfx.dsl.input_resolution.strategies.latest_blessed_model_strategy import LatestBlessedModelStrategy

def init_components(data_dir, transform_module, training_module, tuner_module, training_steps, eval_steps, serving_model_dir):
    output = example_gen_pb2.Output(
        split_config=example_gen_pb2.SplitConfig(splits=[
            example_gen_pb2.SplitConfig.Split(name='train', hash_buckets=8),
            example_gen_pb2.SplitConfig.Split(name='eval', hash_buckets=2)
        ])
    )
    
    example_gen = CsvExampleGen(input_base=data_dir, output_config=output)
    statistics_gen = StatisticsGen(examples=example_gen.outputs['examples'])
    schema_gen = SchemaGen(statistics=statistics_gen.outputs['statistics'])
    example_validator = ExampleValidator(statistics=statistics_gen.outputs['statistics'], schema=schema_gen.outputs['schema'])
    
    transform = Transform(
        examples=example_gen.outputs['examples'],
        schema=schema_gen.outputs['schema'],
        module_file=transform_module
    )
    
    tuner = Tuner(
        module_file=tuner_module,
        examples=transform.outputs['transformed_examples'],
        transform_graph=transform.outputs['transform_graph'],
        schema=schema_gen.outputs['schema'],
        train_args=trainer_pb2.TrainArgs(splits=['train'], num_steps=training_steps),
        eval_args=trainer_pb2.EvalArgs(splits=['eval'], num_steps=eval_steps)
    )
    
    trainer = Trainer(
        module_file=training_module,
        examples=transform.outputs['transformed_examples'],
        transform_graph=transform.outputs['transform_graph'],
        schema=schema_gen.outputs['schema'],
        hyperparameters=tuner.outputs['best_hyperparameters'],
        train_args=trainer_pb2.TrainArgs(splits=['train'], num_steps=training_steps),
        eval_args=trainer_pb2.EvalArgs(splits=['eval'], num_steps=eval_steps)
    )
    
    model_resolver = Resolver(
        strategy_class=LatestBlessedModelStrategy,
        model=Channel(type=Model),
        model_blessing=Channel(type=ModelBlessing)
    ).with_id('latest_blessed_model_resolver')
    
    # PERBAIKAN: Menggunakan label 'Churn' asli untuk Evaluator
    eval_config = tfma.EvalConfig(
        model_specs=[tfma.ModelSpec(label_key='Churn')],
        slicing_specs=[tfma.SlicingSpec()],
        metrics_specs=[
            tfma.MetricsSpec(metrics=[
                tfma.MetricConfig(class_name='ExampleCount'),
                tfma.MetricConfig(class_name='BinaryAccuracy',
                    threshold=tfma.MetricThreshold(
                        value_threshold=tfma.GenericValueThreshold(lower_bound={'value': 0.5}),
                        change_threshold=tfma.GenericChangeThreshold(
                            direction=tfma.MetricDirection.HIGHER_IS_BETTER,
                            absolute={'value': 1e-4}
                        )
                    )
                )
            ])
        ]
    )
    
    evaluator = Evaluator(
        examples=example_gen.outputs['examples'],
        model=trainer.outputs['model'],
        baseline_model=model_resolver.outputs['model'],
        eval_config=eval_config
    )
    
    pusher = Pusher(
        model=trainer.outputs['model'],
        model_blessing=evaluator.outputs['blessing'],
        push_destination=pusher_pb2.PushDestination(
            filesystem=pusher_pb2.PushDestination.Filesystem(base_directory=serving_model_dir)
        )
    )
    
    return (example_gen, statistics_gen, schema_gen, example_validator, transform, tuner, trainer, model_resolver, evaluator, pusher)

Overwriting modules/components.py


In [4]:
%%writefile modules/tuner.py
import keras_tuner as kt
import tensorflow as tf
import tensorflow_transform as tft
from typing import NamedTuple, Dict, Any
from tfx.components.trainer.fn_args_utils import FnArgs

LABEL_KEY = 'Churn'

def transformed_name(key):
    return key + '_xf'

def gzip_reader_fn(filenames):
    return tf.data.TFRecordDataset(filenames, compression_type='GZIP')

def input_fn(file_pattern, tf_transform_output, num_epochs, batch_size=64):
    transform_feature_spec = tf_transform_output.transformed_feature_spec().copy()
    dataset = tf.data.experimental.make_batched_features_dataset(
        file_pattern=file_pattern,
        batch_size=batch_size,
        features=transform_feature_spec,
        reader=gzip_reader_fn,
        num_epochs=num_epochs,
        label_key=transformed_name(LABEL_KEY)
    )
    return dataset

def build_model(hp):
    NUMERIC_FEATURES = ['tenure', 'MonthlyCharges', 'TotalCharges']
    CATEGORICAL_FEATURES = {
        'gender': 2, 'SeniorCitizen': 2, 'Partner': 2,
        'StreamingTV': 3, 'PhoneService': 2, 'InternetService': 3,
        'PaperlessBilling': 2
    }
    
    input_features = []
    for key in NUMERIC_FEATURES:
        input_features.append(tf.keras.Input(shape=(1,), name=transformed_name(key), dtype=tf.float32))
    for key in CATEGORICAL_FEATURES.keys():
        input_features.append(tf.keras.Input(shape=(1,), name=transformed_name(key), dtype=tf.int64))
        
    concatenated_inputs = tf.keras.layers.Concatenate()([tf.cast(inp, tf.float32) for inp in input_features])
    
    dense_units = hp.Int('units', min_value=16, max_value=64, step=16)
    x = tf.keras.layers.Dense(dense_units, activation='relu')(concatenated_inputs)
    x = tf.keras.layers.Dropout(hp.Float('dropout_rate', min_value=0.1, max_value=0.5, step=0.1))(x)
    x = tf.keras.layers.Dense(16, activation='relu')(x)
    outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    
    learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    model = tf.keras.Model(inputs=input_features, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

TunerFnResult = NamedTuple('TunerFnResult', [('tuner', Any), ('fit_kwargs', Dict[str, Any])])

def tuner_fn(fn_args: FnArgs) -> TunerFnResult:
    tf_transform_output = tft.TFTransformOutput(fn_args.transform_graph_path)
    train_dataset = input_fn(fn_args.train_files, tf_transform_output, num_epochs=5)
    eval_dataset = input_fn(fn_args.eval_files, tf_transform_output, num_epochs=5)
    
    tuner = kt.RandomSearch(
        build_model,
        objective='val_accuracy',
        max_trials=3,
        directory=fn_args.working_dir,
        project_name='churn_kt'
    )
    
    fit_kwargs = {
        'x': train_dataset,
        'validation_data': eval_dataset,
        'steps_per_epoch': fn_args.train_steps,
        'validation_steps': fn_args.eval_steps
    }
    
    return TunerFnResult(tuner=tuner, fit_kwargs=fit_kwargs)

Overwriting modules/tuner.py


In [5]:
%%writefile modules/trainer.py
import tensorflow as tf
import tensorflow_transform as tft
from tfx.components.trainer.fn_args_utils import FnArgs
import os

LABEL_KEY = 'Churn'
def transformed_name(key): return key + '_xf'

def gzip_reader_fn(filenames):
    return tf.data.TFRecordDataset(filenames, compression_type='GZIP')

def input_fn(file_pattern, tf_transform_output, num_epochs, batch_size=64):
    transform_feature_spec = tf_transform_output.transformed_feature_spec().copy()
    dataset = tf.data.experimental.make_batched_features_dataset(
        file_pattern=file_pattern,
        batch_size=batch_size,
        features=transform_feature_spec,
        reader=gzip_reader_fn,
        num_epochs=num_epochs,
        label_key=transformed_name(LABEL_KEY)
    )
    return dataset

def build_model(hp):
    NUMERIC_FEATURES = ['tenure', 'MonthlyCharges', 'TotalCharges']
    CATEGORICAL_FEATURES = {
        'gender': 2, 'SeniorCitizen': 2, 'Partner': 2,
        'StreamingTV': 3, 'PhoneService': 2, 'InternetService': 3,
        'PaperlessBilling': 2
    }
    
    input_features = []
    for key in NUMERIC_FEATURES:
        input_features.append(tf.keras.Input(shape=(1,), name=transformed_name(key), dtype=tf.float32))
    for key in CATEGORICAL_FEATURES.keys():
        input_features.append(tf.keras.Input(shape=(1,), name=transformed_name(key), dtype=tf.int64))
        
    concatenated_inputs = tf.keras.layers.Concatenate()([tf.cast(inp, tf.float32) for inp in input_features])
    
    dense_units = hp.get('units') if hp else 32
    dropout_rate = hp.get('dropout_rate') if hp else 0.2
    learning_rate = hp.get('learning_rate') if hp else 1e-3

    x = tf.keras.layers.Dense(dense_units, activation='relu')(concatenated_inputs)
    x = tf.keras.layers.Dropout(dropout_rate)(x)
    x = tf.keras.layers.Dense(16, activation='relu')(x)
    outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    
    model = tf.keras.Model(inputs=input_features, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

def _get_serve_tf_examples_fn(model, tf_transform_output):
    model.tft_layer = tf_transform_output.transform_features_layer()
    @tf.function
    def serve_tf_examples_fn(serialized_tf_examples):
        feature_spec = tf_transform_output.raw_feature_spec()
        feature_spec.pop(LABEL_KEY)
        parsed_features = tf.io.parse_example(serialized_tf_examples, feature_spec)
        transformed_features = model.tft_layer(parsed_features)
        return model(transformed_features)
    return serve_tf_examples_fn

def run_fn(fn_args: FnArgs):
    tf_transform_output = tft.TFTransformOutput(fn_args.transform_output)
    train_dataset = input_fn(fn_args.train_files, tf_transform_output, num_epochs=10)
    eval_dataset = input_fn(fn_args.eval_files, tf_transform_output, num_epochs=10)
    
    hp = fn_args.hyperparameters.get('values') if fn_args.hyperparameters else None
    model = build_model(hp)
    
    model.fit(
        train_dataset,
        steps_per_epoch=fn_args.train_steps,
        validation_data=eval_dataset,
        validation_steps=fn_args.eval_steps,
        epochs=10
    )
    
    signatures = {
        'serving_default': _get_serve_tf_examples_fn(model, tf_transform_output).get_concrete_function(
            tf.TensorSpec(shape=[None], dtype=tf.string, name='examples')
        )
    }
    model.save(fn_args.serving_model_dir, save_format='tf', signatures=signatures)

Overwriting modules/trainer.py


In [6]:
import os
from tfx.orchestration.beam.beam_dag_runner import BeamDagRunner
from tfx.orchestration import metadata, pipeline
from modules.components import init_components

MODULE_ROOT = 'modules'

components = init_components(
    data_dir=DATA_ROOT,
    transform_module=os.path.join(MODULE_ROOT, 'transform.py'),
    training_module=os.path.join(MODULE_ROOT, 'trainer.py'),
    tuner_module=os.path.join(MODULE_ROOT, 'tuner.py'),
    training_steps=400, # diturunkan agar dataset tidak kehabisan data (warning hilang)
    eval_steps=100,
    serving_model_dir=SERVING_MODEL_DIR,
)

pipeline_obj = pipeline.Pipeline(
    pipeline_name=PIPELINE_NAME,
    pipeline_root=PIPELINE_ROOT,
    components=components,
    enable_cache=False,
    metadata_connection_config=metadata.sqlite_metadata_connection_config(METADATA_PATH)
)

BeamDagRunner().run(pipeline_obj)

Trial 3 Complete [00h 00m 02s]
val_accuracy: 0.725781261920929

Best val_accuracy So Far: 0.7809374928474426
Total elapsed time: 00h 00m 07s
Results summary
Results in pipelines\mualim-fth-pipeline\Tuner\.system\executor_execution\55\.temp\55\churn_kt
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 0 summary
Hyperparameters:
units: 32
dropout_rate: 0.4
learning_rate: 0.01
Score: 0.7809374928474426

Trial 1 summary
Hyperparameters:
units: 48
dropout_rate: 0.2
learning_rate: 0.0001
Score: 0.7674999833106995

Trial 2 summary
Hyperparameters:
units: 32
dropout_rate: 0.30000000000000004
learning_rate: 0.0001
Score: 0.725781261920929


Epoch 1/10
400/400 [==============================] - 2s 3ms/step - loss: 0.4470 - accuracy: 0.7871 - val_loss: 0.4588 - val_accuracy: 0.7822
Epoch 2/10
400/400 [==============================] - 1s 2ms/step - loss: 0.4329 - accuracy: 0.7996 - val_loss: 0.4441 - val_accuracy: 0.7848
Epoch 3/10
 62/400 [===>..........................] - ETA: 0s - loss: 0.4385 - accuracy: 0.7933WARNING:tensorflow:Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches (in this case, 4000 batches). You may need to use the repeat() function when building your dataset.


400/400 [==============================] - 2s 6ms/step - loss: 0.4340 - accuracy: 0.7955 - val_loss: 0.4496 - val_accuracy: 0.7877
INFO:tensorflow:struct2tensor is not available.


INFO:tensorflow:struct2tensor is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


...

INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:Assets written to: pipelines\mualim-fth-pipeline\Trainer\model\56\Format-Serving\assets


INFO:tensorflow:Assets written to: pipelines\mualim-fth-pipeline\Trainer\model\56\Format-Serving\assets


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`
